# Calculus, Autodiff & Optimization: Worked Solutions


In [ ]:
import numpy as np
from numpy.testing import assert_allclose
from itertools import product
rng = np.random.default_rng(20260920)


## Problem 1. Compute a Jacobian, JVP, and VJP

**Problem.** For $f(x)=(x_1^2+x_2,x_1x_2)$, compute the numerator-layout Jacobian at $x=(1,2)$. Apply it to input tangent $v=(0.3,-0.4)$ and output cotangent $w=(2,-1)$. Check the directional derivative numerically.

**Solution.** Differentiating each output coordinate gives

$$J_f(x)=\begin{pmatrix}2x_1&1\\x_2&x_1\end{pmatrix},\qquad J_f(1,2)=\begin{pmatrix}2&1\\2&1\end{pmatrix}.$$

The JVP is $Jv=(0.2,0.2)$: if the input follows $x(t)=x+tv$, these are the two output rates $\frac{d}{dt}f(x(t))|_{t=0}$. The VJP is the row $w^TJ=(2,1)$; it differentiates the scalar combination $w^Tf(x)$. Its transpose $J^Tw$ stores the same partials as a column. A central difference is exact in real arithmetic for this quadratic map along a line; the numerical comparison allows floating-point error.


In [ ]:
def explicit_jacobian(x):
    x = np.asarray(x,dtype=float)
    return np.array([[2*x[0],1.],[x[1],x[0]]])


In [ ]:
def vector_function(x):
    return np.array([x[0]**2+x[1],x[0]*x[1]])
x=np.array([1.,2.]); v=np.array([.3,-.4]); w=np.array([2.,-1.])
J=explicit_jacobian(x); eps=1e-5
assert J.shape == (2,2)
assert_allclose(J@v,(vector_function(x+eps*v)-vector_function(x-eps*v))/(2*eps),atol=1e-10)
row_vjp = w[None,:] @ J
column_gradient = J.T @ w[:,None]
assert row_vjp.shape == (1,2) and column_gradient.shape == (2,1)
assert_allclose(row_vjp.T,column_gradient)
assert_allclose(row_vjp,[[2.,1.]])


## Problem 2. Differentiate a graph with shared nodes

**Problem.** Implement scalar reverse-mode differentiation and evaluate $L(x)=(x^2+x)^2$ at $x=2$. Check repeated backward calls and graphs with a shared intermediate.

**Solution.** The forward values are $x^2=4$, $x^2+x=6$, and $L=36$. Reverse accumulation gives $L'(x)=2(x^2+x)(2x+1)$, hence $L'(2)=60$. Multiplication adds a contribution for each input slot; in `x*x`, both contributions reach the same node. A unique-node topological traversal ensures the node's own backward rule runs once after all its consumers have contributed.

This implementation clears every reachable derivative before seeding the output derivative with one, so a second backward call again returns 60. The test $2q^2$ at $q=3$ gives derivative $4q=12$ and checks sharing at a second location. The trigonometric example checks the two paths through $xy$ and $\sin x$.


In [ ]:
class Value:
    def __init__(self, value, parents=()):
        self.value=float(value); self.grad=0.; self.parents=tuple(parents)
        self.local_backward=lambda: None

    @staticmethod
    def coerce(x):
        return x if isinstance(x,Value) else Value(x)

    def __add__(self, other):
        other=self.coerce(other); out=Value(self.value+other.value,(self,other))
        def local():
            self.grad += out.grad
            other.grad += out.grad
        out.local_backward=local
        return out

    __radd__=__add__

    def __mul__(self, other):
        other=self.coerce(other); out=Value(self.value*other.value,(self,other))
        def local():
            self.grad += other.value*out.grad
            other.grad += self.value*out.grad
        out.local_backward=local
        return out

    __rmul__=__mul__

    def sin(self):
        out=Value(np.sin(self.value),(self,))
        def local():
            self.grad += np.cos(self.value)*out.grad
        out.local_backward=local
        return out

    def backward(self):
        topo=[]; visited=set()
        def visit(node):
            if node not in visited:
                visited.add(node)
                for parent in node.parents:
                    visit(parent)
                topo.append(node)
        visit(self)
        for node in topo:
            node.grad=0.
        self.grad=1.
        for node in reversed(topo):
            node.local_backward()


In [ ]:
x=Value(2.)
c=x*x+x; objective=c*c; objective.backward()
assert_allclose(objective.value,36.)
assert_allclose(x.grad,60.)
objective.backward()
assert_allclose(x.grad,60.)  # this engine clears reachable buffers
x=Value(1.); y=Value(2.)
c=x*y+x.sin(); objective=c*c; objective.backward()
assert_allclose(x.grad,2*(2+np.sin(1))*(2+np.cos(1)))
assert_allclose(y.grad,2*(2+np.sin(1)))
q=Value(3.); shared=q*q; out=shared+shared; out.backward()
assert_allclose(q.grad,12.)


## Problem 3. Check a directional derivative

**Problem.** For $f(x)=e^{x_1}+x_1x_2^2$, compare the analytic directional derivative at $x=(0.4,-0.7)$ in direction $v=(0.3,0.8)$ with central differences. Explain the central difference of ReLU at zero.

**Solution.** The gradient is $(e^{x_1}+x_2^2,2x_1x_2)$, so the requested derivative is its inner product with $v$. The function below evaluates $[f(x+\varepsilon v)-f(x-\varepsilon v)]/(2\varepsilon)$ and the checks compare it with that inner product at several step sizes. For the positive coordinate direction, the symmetric difference of $\max(0,x)$ at zero is $(\varepsilon-0)/(2\varepsilon)=1/2$. ReLU has no classical derivative at zero, so this number does not contradict an automatic-differentiation convention that returns zero there.


In [ ]:
def central_direction(f, x, v, eps=1e-5):
    x, v = np.asarray(x,dtype=float), np.asarray(v,dtype=float)
    return (f(x+eps*v)-f(x-eps*v))/(2*eps)


In [ ]:
x=np.array([.4,-.7]);v=np.array([.3,.8])
f=lambda z: np.exp(z[0])+z[0]*z[1]**2
grad=np.array([np.exp(x[0])+x[1]**2,2*x[0]*x[1]])
assert_allclose(central_direction(f,x,v),grad@v,rtol=1e-8,atol=1e-9)
errors=[abs(central_direction(f,x,v,e)-grad@v) for e in [1e-2,1e-3,1e-4]]
assert errors[1] < errors[0] and errors[2] < errors[1]
relu=lambda z: max(0.,z[0])
assert_allclose(central_direction(relu,[0.],[1.]),.5)
# The symmetric difference at a kink need not equal an AD convention.


## Problem 4. Derive the batched affine pullback

**Problem.** For $Z=XW^T+\mathbf1b^T$ and a supplied output gradient $\Delta=\nabla_ZL$, compute gradients with respect to $X,W,b$. Verify the differential identity on arbitrary perturbations.

**Solution.** Expanding $dZ=dXW^T+XdW^T+\mathbf1db^T$ in $dL=\operatorname{tr}(\Delta^TdZ)$ yields

$$\nabla_XL=\Delta W,\qquad \nabla_WL=\Delta^TX,\qquad \nabla_bL=\Delta^T\mathbf1.$$

These have the shapes of their respective arguments. The given $\Delta$ already contains any factor from the loss reduction, so no additional batch division appears. For the supplied array its column sums, hence the bias gradient, are $(0.2,2.2)$. The numerical identity compares the inner product of $\Delta$ with a perturbed output against the sum of the three input-gradient inner products.


In [ ]:
def affine_pullback(X, W, delta):
    return delta@W, delta.T@X, delta.sum(axis=0)


In [ ]:
X=np.array([[1.,-2.],[3.,.5],[-1.,2.]])
W=np.array([[2.,-1.],[.3,.7]])
b=np.array([.1,-.2]);delta=np.array([[1.,2.],[-1.,.5],[.2,-.3]])
dX,dW,db=affine_pullback(X,W,delta)
assert dX.shape==X.shape and dW.shape==W.shape and db.shape==b.shape
assert_allclose(db,[.2,2.2])
u=rng.normal(size=X.shape); v=rng.normal(size=W.shape); q=rng.normal(size=b.shape)
perturbed_output=u@W.T + X@v.T + q
left=np.sum(delta*perturbed_output)
right=np.sum(dX*u)+np.sum(dW*v)+db@q
assert_allclose(left,right,atol=1e-12)


## Problem 5. Backpropagate through a two-layer composition

**Problem.** With row batches, let $A=XW_1^T+b_1$, $H=\tanh A$, $Z=HW_2^T+b_2$, and $L=\|Z-Y\|_F^2/(2B)$. Compute gradients of both weights, both biases, and the input. Verify them by directional differences and PyTorch, then compare duplicated batches and the alternative reduction $\widetilde L=L/k$.

**Solution.** The output gradient is $\Delta_2=(Z-Y)/B$. Applying the affine pullback yields $\nabla_HL=\Delta_2W_2$, $\nabla_{W_2}L=\Delta_2^TH$, and the output-bias column sum. Because $\tanh'(A)=1-H^2$, the hidden gradient is $\Delta_1=(\Delta_2W_2)\odot(1-H^2)$. A second affine pullback returns the remaining three gradients.

Duplicating every row doubles both the residual sum and the batch divisor. The loss and all parameter gradients remain unchanged; each copied input row has half the former gradient. Summing the two copies would recover the derivative with respect to an original shared input. Replacing the reduction by $1/(2Bk)$ divides the loss and every gradient by the output width $k$, once only. The checks below verify both identities as well as all five gradient arrays.


In [ ]:
def two_layer_value_and_grads(X, W1, b1, W2, b2, Y):
    A=X@W1.T+b1; H=np.tanh(A); Z=H@W2.T+b2
    residual=Z-Y; loss=np.sum(residual**2)/(2*len(X))
    delta2=residual/len(X)
    dH,dW2,db2=affine_pullback(H,W2,delta2)
    delta1=dH*(1-H**2)
    dX,dW1,db1=affine_pullback(X,W1,delta1)
    return loss,(dX,dW1,db1,dW2,db2)


In [ ]:
X=rng.normal(size=(4,3)); W1=rng.normal(size=(5,3))*.3
b1=rng.normal(size=5)*.1;W2=rng.normal(size=(2,5))*.3
b2=rng.normal(size=2)*.1;Y=rng.normal(size=(4,2))
args=[X,W1,b1,W2,b2]
loss,grads=two_layer_value_and_grads(*args,Y)
for j,(a,g) in enumerate(zip(args,grads)):
    direction=rng.normal(size=a.shape);eps=1e-5
    plus=[t.copy() for t in args];minus=[t.copy() for t in args]
    plus[j]+=eps*direction;minus[j]-=eps*direction
    numeric=(two_layer_value_and_grads(*plus,Y)[0]-two_layer_value_and_grads(*minus,Y)[0])/(2*eps)
    assert_allclose(numeric,np.sum(g*direction),rtol=1e-7,atol=1e-9)
loss2,grads2=two_layer_value_and_grads(np.concatenate([X,X]),W1,b1,W2,b2,np.concatenate([Y,Y]))
assert_allclose(loss2,loss)
for g,g2 in zip(grads[1:],grads2[1:]):
    assert_allclose(g,g2,atol=1e-14)
assert_allclose(grads2[0][:len(X)]*2,grads[0])
assert_allclose(grads2[0][len(X):]*2,grads[0])


In [ ]:
import torch
# Coordinate arrays use double precision to make the numerical comparison sharp.
tensors=[torch.tensor(a,dtype=torch.float64,requires_grad=True) for a in args]
tX,tW1,tb1,tW2,tb2=tensors
tY=torch.tensor(Y,dtype=torch.float64)
tZ=torch.tanh(tX@tW1.T+tb1)@tW2.T+tb2
tloss=((tZ-tY)**2).sum()/(2*len(X))
tgrads=torch.autograd.grad(tloss,tensors)
assert_allclose(tloss.detach().numpy(),loss)
for manual,automatic in zip(grads,tgrads):
    assert_allclose(manual,automatic.detach().numpy(),rtol=1e-11,atol=1e-12)
layer=torch.nn.Linear(3,5,dtype=torch.float64)
assert tuple(layer.weight.shape)==(5,3)
print("Every weight, bias, and input gradient matches PyTorch.")


In [ ]:
# Averaging over all B*k outputs introduces exactly one factor 1/k.
k = Y.shape[1]
scaled_loss = loss / k
scaled_grads = tuple(g / k for g in grads)
tensors_scaled = [torch.tensor(a, dtype=torch.float64, requires_grad=True) for a in args]
sX, sW1, sb1, sW2, sb2 = tensors_scaled
sZ = torch.tanh(sX @ sW1.T + sb1) @ sW2.T + sb2
entry_mean_loss = 0.5 * ((sZ - tY)**2).mean()
entry_mean_grads = torch.autograd.grad(entry_mean_loss, tensors_scaled)
assert_allclose(entry_mean_loss.detach().numpy(), scaled_loss)
for manual, automatic in zip(scaled_grads, entry_mean_grads):
    assert_allclose(manual, automatic.detach().numpy(), rtol=1e-11, atol=1e-12)


## Problem 6. Solve quadratic gradient-descent dynamics

**Problem.** Starting from $(1,1)$, apply constant-step gradient descent to $F(x)=x^T\operatorname{diag}(1,20)x/2$ with steps $0.05,0.1,0.12$.

**Solution.** The coordinates are $x_{t,1}=(1-\eta)^t$ and $x_{t,2}=(1-20\eta)^t$. Convergence from every initial point requires $0<\eta<0.1$. Step $0.05$ gives first coordinate $0.95^t$ and zeros the second after one update. At $0.1$, the second coordinate is $(-1)^t$ and never decays. At $0.12$, it is $(-1.4)^t$ and diverges in magnitude. The larger curvature restricts stability; the smaller curvature gives the slow contraction for the stable step used here.


In [ ]:
H=np.diag([1.,20.]); initial=np.ones(2)
def quadratic_path(eta,steps=60):
    points=[initial.copy()]
    for _ in range(steps):
        points.append(points[-1]-eta*(H@points[-1]))
    return np.array(points)
stable=quadratic_path(.05);boundary=quadratic_path(.1);unstable=quadratic_path(.12)
assert np.linalg.norm(stable[-1]) < np.linalg.norm(initial)
assert_allclose(boundary[:,1],(-1.)**np.arange(len(boundary)))
assert abs(unstable[-1,1]) > 1e8
assert_allclose(stable[:,0],.95**np.arange(len(stable)))


## Problem 7. Enumerate minibatch gradients

**Problem.** At a fixed parameter, let the three component gradients be $(-2,1)$, $(1,-1)$, and $(4,3)$. Enumerate every ordered batch of size two sampled uniformly with replacement. Verify the mean and covariance of their averaged gradients.

**Solution.** The component mean is $(1,1)$, and the deviations are $(-3,0)$, $(0,-2)$, and $(3,2)$. Their covariance, using divisor three, is

$$\Sigma=\frac13\begin{pmatrix}18&6\\6&8\end{pmatrix}=\begin{pmatrix}6&2\\2&8/3\end{pmatrix}.$$

There are $3^2=9$ equally likely ordered batches. Each averaged gradient has expectation $(1,1)$; independence of the two draws makes its covariance $\Sigma/2$. Repeated indices are valid draws and must retain both contributions. The enumeration verifies these exact finite-distribution identities rather than estimating them from a random subsample.


In [ ]:
def minibatch_mean(component_gradients, indices):
    return np.asarray(component_gradients)[np.asarray(indices,dtype=int)].mean(axis=0)


In [ ]:
components=np.array([[-2.,1.],[1.,-1.],[4.,3.]])
full=components.mean(axis=0)
batchmeans=np.stack([minibatch_mean(components,idx) for idx in product(range(3),repeat=2)])
assert_allclose(batchmeans.mean(axis=0),full)
centered=components-full
sigma=centered.T@centered/len(components)
batch_centered=batchmeans-full
assert_allclose(batch_centered.T@batch_centered/len(batchmeans),sigma/2)
assert_allclose(minibatch_mean(components,[0,0]),components[0])


## Problem 8. Continue a momentum state

**Problem.** Implement $v'=\beta v+g$, $\theta'=\theta-\eta v'$ without mutating the input arrays. For $\beta=0.8$ and constant gradient $g$, find the first three velocities from zero. Compare an uninterrupted quadratic trajectory with a resumed trajectory and a reset-velocity trajectory.

**Solution.** The velocities are $g$, $1.8g$, and $2.44g$. For scalar gradient two these are $2,3.6,4.88$. Returning both the parameter and velocity preserves the complete state required by this update rule. Seven updates followed by thirteen from the saved pair therefore reproduce twenty uninterrupted updates exactly. Resetting only the velocity omits past-gradient contributions and changes the continuation. At $\beta=0$, the update reduces to ordinary gradient descent, irrespective of the supplied old velocity.


In [ ]:
def momentum_step(theta, velocity, gradient, eta, beta):
    new_velocity=beta*np.asarray(velocity)+np.asarray(gradient)
    return np.asarray(theta)-eta*new_velocity,new_velocity


In [ ]:
theta=np.array([1.,-2.]);velocity=np.zeros(2);g=np.array([2.,-1.])
theta_copy=theta.copy();v_copy=velocity.copy()
a,v=momentum_step(theta,velocity,g,.1,.8)
assert_allclose(a,theta-.1*g);assert_allclose(v,g)
assert_allclose(theta,theta_copy);assert_allclose(velocity,v_copy)
_,v2=momentum_step(a,v,g,.1,.8)
_,v3=momentum_step(a,v2,g,.1,.8)
assert_allclose(v3,2.44*g)
H=np.diag([1.,3.])
def run_state(state,steps):
    x,v=map(np.copy,state)
    for _ in range(steps):
        x,v=momentum_step(x,v,H@x,.05,.8)
    return x,v
start=(np.array([1.,-2.]),np.zeros(2))
mid=run_state(start,7); continued=run_state(mid,13); whole=run_state(start,20)
for a,b in zip(continued,whole):assert_allclose(a,b,atol=0.,rtol=0.)
reset=run_state((mid[0],np.zeros(2)),13)
assert not np.allclose(reset[0],whole[0])
x,v=momentum_step(start[0],np.array([4.,5.]),H@start[0],.05,0.)
assert_allclose(x,start[0]-.05*H@start[0])
